# Memory tests: the associative matrix as a store (Track A), and memory layers inside Hyena (Track B)

Protocol, decision rules and predictions, written before any run: `memtest/PROTOCOL.md`.

- **Track A** (CPU work, minutes to about an hour): is linear attention's fixed matrix a useful store? It is compared with FAISS at **equal memory** (PQ), plus HNSW and exact search as ceilings. Results: `/content/results/store.jsonl`.
- **Track B** (GPU work, the long part): does a cheap memory layer (delta rule, or a lookup over chunk summaries) give a Hyena model long-range recall like one attention layer, with less memory? Results: `/content/results/recall.jsonl`.

**Use an A100 runtime** for the whole notebook (Track A runs on its CPU). Times are not measured yet: check Track B's pilot cell before its main stage. Finished stages and runs are skipped on re-run; after a deleted runtime set `RESTORE = True` and upload `results.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, json, subprocess, sys
REPO, DATA, OUTDIR = '/content/grassman', '/content/data', '/content/results'
RUNS = f'{OUTDIR}/store.jsonl'
os.makedirs(OUTDIR, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/thebnbrkr/grassman.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q datasets transformers
# drop any grassman modules imported earlier in this runtime, so the code just pulled is what runs
for _name in [k for k in sys.modules if k == 'grassman' or k.startswith('grassman.')]:
    del sys.modules[_name]

In [ ]:
!pip install -q faiss-cpu sentence-transformers

Only if you are **resuming this notebook** in a new runtime: set `RESTORE = True` to upload the `results.zip` it downloaded earlier, so finished runs are skipped. On a first run leave it `False`.

In [ ]:
RESTORE = False
if RESTORE:
    from google.colab import files
    for name in files.upload():
        !cd /content && unzip -o -q "{name}"
!ls -la {OUTDIR}

Unit tests (about a minute). Stop if anything fails.

In [ ]:
!python -m pytest -q

In [ ]:
STORE_RUNS, RECALL_RUNS = f'{OUTDIR}/store.jsonl', f'{OUTDIR}/recall.jsonl'

# Track A: the matrix as a store

In [ ]:
EMB = '/content/data/quora_emb.npz'
def stage(name, args):
    marker = f'{OUTDIR}/store_{name}.done'
    if os.path.exists(marker):
        print('skip', name, '(done)'); return
    subprocess.run([sys.executable, '-m', 'memtest.store', *args, '--out', STORE_RUNS, '--emb', EMB], cwd=REPO, check=True)
    open(marker, 'w').close()

Embed the Quora pairs once (dedupe, split into calibration / dev / test, 20,000 pairs each).

In [ ]:
if not os.path.exists(EMB):
    subprocess.run([sys.executable, '-m', 'memtest.store', 'embed', '--emb', EMB], cwd=REPO, check=True)

## A1: capacity on random keys

In [ ]:
stage('capacity', ['capacity'])

## A2: real data. Dev first (the matrix variant is chosen here), then test (used once).

In [ ]:
stage('real_dev', ['real', '--split', 'dev'])

In [ ]:
stage('real_test', ['real', '--split', 'test'])

## A3: updates and deletes

In [ ]:
stage('updates', ['updates'])

## A5: cost against N

In [ ]:
stage('cost', ['cost'])

## Results

In [ ]:
out = subprocess.run([sys.executable, '-m', 'memtest.store', 'summary', '--out', STORE_RUNS], cwd=REPO,
                     capture_output=True, text=True, check=True).stdout
open(f'{OUTDIR}/store_summary.md', 'w').write(out)
from IPython.display import Markdown, display
display(Markdown(out))

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np
from collections import defaultdict
rows = [json.loads(l) for l in open(STORE_RUNS) if '"exp": "real"' in l]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, dv in zip(axes, (64, 384, 1536)):
    by = defaultdict(list)
    for r in rows:
        if r['split'] == 'test' and r['dv'] == dv and r['recall1'] is not None:
            by[r['method']].append((r['N'], r['recall1']))
    for m in ('add+c', 'delta+c', 'lstsq+c', 'pq', 'fifo', 'flat'):
        if by[m]:
            Ns = sorted({n for n, _ in by[m]})
            ax.plot(Ns, [100 * np.mean([v for n, v in by[m] if n == N]) for N in Ns], marker='o', label=m)
    ax.set_xscale('log'); ax.set_title(f'test, {384 * dv * 4 / 1e3:.0f} KB'); ax.set_xlabel('items stored (N)')
    ax.grid(alpha=.3)
axes[0].set_ylabel('recall@1 %'); axes[0].legend()
plt.savefig(f'{OUTDIR}/store_real.png', dpi=150, bbox_inches='tight'); plt.show()

## How to read it (decision rules from `memtest/PROTOCOL.md`)

- **Dropped as a store** if the chosen matrix variant is worse than PQ at N = 384 and N = 768 (Holm p < 0.05).
- **"Useful memory tier"** only if it is within 2 points of PQ, or better, at every N ≥ 384.
- `flat` and `hnsw` have no budget: they show the best any method could do, not a fair comparison.

# Track B: memory layers inside Hyena

In [ ]:
def train(model, task, T, pairs, seed, lr, steps, batch, tag, test=False):
    cmd = [sys.executable, '-m', 'memtest.train', '--model', model, '--task', task, '--T', str(T),
           '--pairs', str(pairs), '--seed', str(seed), '--lr', str(lr), '--steps', str(steps),
           '--batch', str(batch), '--tag', tag, '--out', RECALL_RUNS, '--skip-existing']
    if not test:
        cmd += ['--no-test']             # pilot and LR sweep never touch the test set
    subprocess.run(cmd, cwd=REPO, check=True)

def analyze(*args):
    print(subprocess.run([sys.executable, '-m', 'memtest.analyze', RECALL_RUNS, *map(str, args)], cwd=REPO,
                         capture_output=True, text=True, check=True).stdout)

In [ ]:
MAIN = ['hyena', 'hyena+attn', 'hyena+linear', 'hyena+delta', 'hyena+moment', 'attention', 'swa']
PLACEMENT = ['hyena+attn@0', 'hyena+attn@3']
STEPS = 8000
# same tokens per step at every length: 64 x 1024 = 16 x 4096 = 128 x 512
BATCH = {512: 128, 1024: 64, 4096: 16}

## Stage 0: pilot (dev only)

Two models, 2,000 steps. **Check before going on:** `attention` should reach > 90% dev accuracy; if it does not, the step count is too low for this task (raise `STEPS` and note it in the log). The printed time per run tells you how long the rest will take (there are 21 sweep runs and 63 main runs).

In [ ]:
import time
for m in ['attention', 'hyena']:
    t0 = time.time()
    train(m, 'far', 1024, 8, 0, 1e-3, 2000, BATCH[1024], 'pilot')
    print(f'{m}: {time.time() - t0:.0f} s for 2,000 steps')
analyze('--tag', 'pilot', '--dev')

## Stage 1: learning-rate sweep (far, T = 1024, seed 0, dev only)

In [ ]:
LRS = [3e-4, 1e-3, 3e-3]
for m in MAIN + PLACEMENT:
    for lr in LRS:
        train(m, 'far', 1024, 8, 0, lr, STEPS, BATCH[1024], 'sweep')
best = json.loads(subprocess.check_output([sys.executable, '-m', 'memtest.analyze', RECALL_RUNS, '--tag', 'sweep',
                                           '--best-lr'], cwd=REPO, text=True))
print('best learning rate per model:', best)
analyze('--tag', 'sweep', '--dev')

## Stage 2: main runs (3 seeds, test set once)

Seed-major, so an interruption still leaves complete seeds. The number of seeds is fixed by the protocol: do not add seeds after looking at results.

In [ ]:
SEEDS = [0, 1, 2]
for seed in SEEDS:
    for m in MAIN:
        train(m, 'far', 1024, 8, seed, best[m], STEPS, BATCH[1024], 'main', test=True)
        train(m, 'far', 4096, 8, seed, best[m], STEPS, BATCH[4096], 'main', test=True)
    for m in MAIN + PLACEMENT:
        train(m, 'mqar', 512, 64, seed, best[m], STEPS, BATCH[512], 'main', test=True)

## Results

**Primary (confirmatory):** far recall at T = 4096, queries ≥ 1024 tokens back.

In [ ]:
analyze('--tag', 'main', '--task', 'far', '--T', 4096, '--confirmatory', '--min-dist', 1024)

Secondary (exploratory): far at T = 1024, and MQAR (all distances).

In [ ]:
analyze('--tag', 'main', '--task', 'far', '--T', 1024, '--min-dist', 256)

In [ ]:
analyze('--tag', 'main', '--task', 'mqar', '--T', 512, '--min-dist', 0)

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from memtest import analyze as A, tasks
rows = A.load(RECALL_RUNS, 'main', 'far', 4096)
fig, ax = plt.subplots(figsize=(8, 4.5))
mids = [np.sqrt(max(1, a) * b) for a, b in zip(tasks.BUCKETS[:-1], tasks.BUCKETS[1:])]
for name, rs in A._group(rows).items():
    c = np.sum([r['test_detail']['bucket_correct'] for r in rs], 0)
    t = np.sum([r['test_detail']['bucket_total'] for r in rs], 0)
    keep = t > 0
    ax.plot(np.array(mids)[keep], 100 * c[keep] / t[keep], marker='o', label=f'{name} ({len(rs)} seeds)')
ax.set_xscale('log'); ax.set_xlabel('distance from key to query (tokens)'); ax.set_ylabel('test accuracy %')
ax.set_title('far recall, T = 4096'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.savefig(f'{OUTDIR}/recall_by_distance.png', dpi=150, bbox_inches='tight'); plt.show()

## How to read it (decision rules from `memtest/PROTOCOL.md`)

- **"Moment memory adds recall":** hyena+moment − hyena positive, Holm p < 0.05, 3/3 seeds.
- **"Matches attention":** the 95% interval of hyena+moment − hyena+attn lies within ±5 points.
- The `state (MB, bf16)` column is the memory each model keeps per sequence to generate the next token; that is the trade-off this test is about.

Download the results before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')